In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

# Lecture Bronze
df = spark.table("pharma_catalog.bronze.bronze_calendar")
display(df)

In [0]:
# COMMAND ----------
# Déduplication
df_dedup = df.dropDuplicates(["date_key"])
print(f"Avant: {df.count()} | Après dédup: {df_dedup.count()}")

# Contrôles qualité
df_quality = df_dedup.withColumn("rule_failed",
    F.when(F.col("date_key").isNull(), F.lit("null_date_key"))
     .when(F.col("full_date").isNull(), F.lit("null_full_date"))
     .when(F.col("day").isNull() | (F.col("day") < 1) | (F.col("day") > 31), F.lit("jour_invalide"))
     .when(F.col("month").isNull() | (F.col("month") < 1) | (F.col("month") > 12), F.lit("mois_invalide"))
     .when(F.col("quarter").isNull() | (F.col("quarter") < 1) | (F.col("quarter") > 4), F.lit("trimestre_invalide"))
     .when(F.col("year").isNull(), F.lit("null_year"))
     .when(F.col("week_number").isNull() | (F.col("week_number") < 1) | (F.col("week_number") > 53), F.lit("semaine_invalide"))
     .otherwise(F.lit(None))
)

passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f" Lignes OK : {passed.count()}")
print(f" Lignes rejetées : {failed.count()}")
display(failed)

In [0]:
# COMMAND ----------
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_calendar")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))
    df_quarantine.write.mode("append") \
                       .option("mergeSchema", "true") \
                       .saveAsTable("pharma_catalog.silver_quarantine.quarantine")
else:
    print(" Aucune ligne rejetée — quarantine vide")





In [0]:
# COMMAND ----------
passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_calendar")
print(f" {passed.count()} ligne(s) chargée(s) dans silver_calendar")

In [0]:
# COMMAND ----------
display(spark.table("pharma_catalog.silver.silver_calendar"))

# COMMAND ----------
display(spark.table("pharma_catalog.silver_quarantine.quarantine"))